# TTS API บน Google Colab (VoxCPM2 / OmniVoice)

1. Runtime → Change runtime type → **T4 GPU**
2. กด Runtime → **Run all** (Ctrl+F9) แล้วกด Run anyway
3. ราว 10 วินาที ช่อง 1 จะขึ้นปุ่มสีน้ำเงิน **"คลิกที่นี่ 1 ครั้ง: ส่ง URL/KEY ให้ Dub Studio"** → คลิก (Dub Studio V4 ต้องเปิดอยู่บนเครื่องนี้) · ใช้ `client.py` ก็ก๊อป `URL` กับ `KEY` ไปใส่เอง
4. รอช่อง 2 โหลดโมเดล (ครั้งแรกหลายนาที) — Dub Studio รอเองแล้วเดินงานต่อ

`ENGINE` ในช่อง 1: `voxcpm` = VoxCPM2 (weights Apache-2.0 ใช้เชิงพาณิชย์ได้) · `omnivoice` = weights CC-BY-NC ห้ามใช้เชิงพาณิชย์

ปิดแท็บ/หลุด runtime = URL ตาย ต้อง Run all ใหม่ (URL ใหม่ทุกรอบ) · ช่อง 6 รันค้างไว้ตลอด (เฝ้าเซิร์ฟเวอร์ + กัน Colab ตัดเพราะว่าง) — เป็นเรื่องปกติ ไม่ต้องกดหยุด


In [ ]:
# 1) เปิดอุโมงค์ก่อน (ได้ลิงก์ส่งค่าให้ Dub Studio ภายใน ~10 วิ) แล้วดึงโค้ดจาก GitHub + ติดตั้งโมเดล
ENGINE = "voxcpm"   # "voxcpm" (VoxCPM2) หรือ "omnivoice"
REPO = "https://github.com/zannbls-bot/omnivoice-colab"
import os, re, json, secrets, subprocess, time, urllib.request
from IPython.display import display, HTML
KEY = secrets.token_urlsafe(16)
![ -x /content/cloudflared ] || (wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared)

def start_tunnel():
    global tun, URL
    subprocess.run(["pkill", "-f", "cloudflared tunnel"])
    tun = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
                           stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)
    URL = None
    for _ in range(60):
        m = re.search(r"https://(?!api\.)[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
        if m: URL = m.group(0); break
        time.sleep(2)
    print("URL =", URL)
    print("KEY =", KEY)
    # Dub Studio บนเครื่องนี้รับค่าเองเมื่อคลิกลิงก์ (เบราว์เซอร์ไม่ให้หน้านี้ส่งเข้าเครื่องเองโดยไม่มีการคลิก)
    display(HTML(f'<a href="http://127.0.0.1:8765/colab#u={URL}&k={KEY}" target="_blank" '
                 'style="display:inline-block;margin:8px 0;padding:12px 18px;background:#2563eb;color:#fff;'
                 'border-radius:10px;font:600 16px sans-serif;text-decoration:none">'
                 '➡ คลิกที่นี่ 1 ครั้ง: ส่ง URL/KEY ให้ Dub Studio บนเครื่องนี้</a>'))

start_tunnel()
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf /content/ovapi && git clone -q --depth 1 $REPO /content/ovapi
!pip install -q $ENGINE   # package name = engine name

In [ ]:
# 1b) งานหนักของ Dub Studio: แยกเสียงพูด (BS-Roformer) · แยกเอฟเฟกต์ (Bandit-Plus DnR แบบโปรเจค 3-3) · ถอดซับ (faster-whisper large-v3)
!pip install -q "audio-separator[gpu]" faster-whisper ml_collections omegaconf spafe asteroid pytorch_lightning torchmetrics pedalboard loralib
![ -d /content/MSST ] || git clone -q --depth 1 https://github.com/ZFTurbo/Music-Source-Separation-Training /content/MSST
![ -s /content/MSST/model_bandit_plus_dnr_sdr_11.47.chpt ] || wget -q -O /content/MSST/model_bandit_plus_dnr_sdr_11.47.chpt https://github.com/ZFTurbo/Music-Source-Separation-Training/releases/download/v.1.0.3/model_bandit_plus_dnr_sdr_11.47.chpt
!ls -la /content/MSST/model_bandit_plus_dnr_sdr_11.47.chpt

In [ ]:
# 2) เปิดเซิร์ฟเวอร์ (ครั้งแรกโหลดโมเดลหลายนาที)
SERVER = {"voxcpm": "server_voxcpm.py", "omnivoice": "server.py"}[ENGINE]

def start_server():
    global srv
    # rerun = stop the old server first. -9: a server that got a plain TERM waits for its running jobs (minutes) and keeps the port
    subprocess.run(["pkill", "-9", "-f", "ovapi/server"]); time.sleep(3)
    srv = subprocess.Popen(["python", "/content/ovapi/" + SERVER], env=dict(os.environ, OV_KEY=KEY, OV_PORT="8000"),
                           stdout=open("/content/server.log", "a"), stderr=subprocess.STDOUT)
    for _ in range(180):
        try:
            print(urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=5).read().decode()); return
        except Exception:
            if srv.poll() is not None:
                print(open("/content/server.log").read()[-3000:]); raise SystemExit("server died")
            time.sleep(5)
    print(open("/content/server.log").read()[-3000:]); raise SystemExit("server not ready in 15 min")

start_server()

In [ ]:
# 3) พร้อมใช้ — URL/KEY ชุดเดียวกับช่อง 1 (Dub Studio: คลิกลิงก์สีน้ำเงินในช่อง 1 ถ้ายังไม่ได้คลิก · client.py: ก๊อปสองค่านี้)
print("URL =", URL)
print("KEY =", KEY)
print()
print(f'python client.py --url {URL} --key {KEY} --language th --text "สวัสดีครับ ทดสอบเสียง" --out test.wav')

In [ ]:
# 4) (ไม่บังคับ) ทดสอบในนี้เลย
from IPython.display import Audio
import json
req = urllib.request.Request("http://127.0.0.1:8000/tts",
    json.dumps({"text": "สวัสดีครับ ทดสอบเสียงจาก Colab", "language": "th"}).encode(),
    {"Content-Type": "application/json", "X-API-Key": KEY})
open("/content/t.wav", "wb").write(urllib.request.urlopen(req).read())
Audio("/content/t.wav")

In [ ]:
# 5) ดู log เซิร์ฟเวอร์ (เวลามีปัญหา)
print(open("/content/server.log").read()[-3000:])

In [ ]:
# 6) เฝ้าเซิร์ฟเวอร์ — รันค้างไว้ตลอด อย่ากดหยุด: ช่องนี้ทำงานอยู่ = Colab ไม่ตัดเพราะว่าง · เซิร์ฟเวอร์/อุโมงค์ตาย = เปิดใหม่เอง
#    ไม่มีงานเข้ามาเกิน IDLE_MIN นาที = คืนเครื่อง Colab เอง (ประหยัดโควตาการ์ดจอ · Dub Studio ต่อใหม่ให้เองเมื่อมีงาน) · 0 = ไม่คืน
IDLE_MIN = 30
import datetime, json
n = 0
while True:
    time.sleep(30); n += 1
    now = datetime.datetime.now().strftime("%H:%M")
    if srv.poll() is not None:
        print(now, "server died -> restart"); start_server()
    if tun.poll() is not None:
        print(now, "tunnel died -> restart (new URL)"); start_tunnel()
    try:
        h = json.loads(urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=20).read())
    except Exception:
        h = {}
    if IDLE_MIN and h.get("idle", 0) > IDLE_MIN * 60:
        print(now, f"no jobs for {IDLE_MIN} min -> releasing the Colab runtime (saves GPU quota)", flush=True)
        from google.colab import runtime
        runtime.unassign()
    if n % 60 == 0:
        print(now, "ok · idle", h.get("idle"), "s · up", h.get("up"), "s", flush=True)